In [1]:
import os
import torch
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from mpstemmer import MPStemmer

import gensim
from gensim import corpora
from gensim.utils import simple_preprocess
from pprint import pprint

from transformers import BertTokenizer
from nltk.tokenize import RegexpTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print("Total GPU:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

Total GPU: 1
Current GPU: NVIDIA RTX A5000


In [2]:
path = "../bert_data/id-p2"

train_files = []
val_files = []
test_files = []
for file in os.listdir(path):
    if "bert.pt" in file and "train" in file:
        train_files.append(path + "/" + file)
    elif "bert.pt" in file and "valid" in file:
        val_files.append(path + "/" + file)
    elif "bert.pt" in file and "test" in file:
        test_files.append(path + "/" + file)

train_files = sorted(train_files)
val_files = sorted(val_files)
test_files = sorted(test_files)

In [3]:
train_files = train_files[:]
val_files = val_files[:]
test_files = test_files[:]

In [4]:
train_docs = []

i = 0
for file in train_files:
    print(f"Loading data train {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        train_docs.append(src)
    i = i + 1

Loading data train 0...
Loading data train 1...
Loading data train 2...
Loading data train 3...
Loading data train 4...
Loading data train 5...
Loading data train 6...
Loading data train 7...
Loading data train 8...
Loading data train 9...
Loading data train 10...
Loading data train 11...
Loading data train 12...
Loading data train 13...
Loading data train 14...
Loading data train 15...
Loading data train 16...
Loading data train 17...
Loading data train 18...
Loading data train 19...


In [5]:
val_docs = []

i = 0
for file in val_files:
    print(f"Loading data val {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        val_docs.append(src)
    i = i + 1

Loading data val 0...
Loading data val 1...
Loading data val 2...


In [6]:
test_docs = []

i = 0
for file in test_files:
    print(f"Loading data test {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        test_docs.append(src)
    i = i + 1

Loading data test 0...
Loading data test 1...
Loading data test 2...


In [7]:
# Remove stop words
def remove_stop_words(doc):
    factory = StopWordRemoverFactory()
    stopword = factory.create_stop_word_remover()
    res = stopword.remove(doc)
    return res

In [8]:
rm_train_docs = []
rm_val_docs = []
rm_test_docs = []

# Preprocess only removing stop words
# If we also use stemming, the topic will be non-sense
for doc in train_docs:
    # print("Preprocess train docs...")
    doc = remove_stop_words(doc)
    rm_train_docs.append(doc)

for doc in val_docs:
    # print("Preprocess val docs...")
    doc = remove_stop_words(doc)
    rm_val_docs.append(doc)

for doc in test_docs:
    # print("Preprocess test docs...")
    doc = remove_stop_words(doc)
    rm_test_docs.append(doc)

In [9]:
def tokenize(docs):
    # Split the documents into tokens.
    tokenizer = RegexpTokenizer(r'\w+')
    new_docs = docs.copy()
    for idx in range(len(docs)):
        new_docs[idx] = docs[idx].lower()  # Convert to lowercase.
        new_docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words.
        
    return new_docs

In [10]:
proc_train_docs = tokenize(rm_train_docs)
proc_val_docs = tokenize(rm_train_docs)
proc_test_docs = tokenize(rm_train_docs)

In [11]:
print("Saving docs...")
torch.save(proc_train_docs, "./lda/proc_train_doc.pt")
torch.save(proc_val_docs, "./lda/proc_val_doc.pt")
torch.save(proc_test_docs, "./lda/proc_test_doc.pt")

Saving docs...


In [12]:
# Create a dictionary representation of the documents.
dictionary = corpora.Dictionary(proc_train_docs)

In [13]:
# Bag-of-words representation of the documents.
corpus = [dictionary.doc2bow(doc) for doc in proc_train_docs]
val_bow = [dictionary.doc2bow(doc) for doc in proc_val_docs]
test_bow = [dictionary.doc2bow(doc) for doc in proc_test_docs]

In [14]:
print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

Number of unique tokens: 182143
Number of documents: 38207


In [15]:
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                   level=logging.DEBUG,
                   filename='lda_model.log')

In [16]:
# Train LDA model.
from gensim.models import LdaModel

# Set training parameters.
num_topics = 200
chunksize = 5000
passes = 1
iterations = 400
eval_every = None  # Don't evaluate model perplexity, takes too much time.

# Make an index to word dictionary.
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary.id2token

model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    chunksize=chunksize,
    alpha='auto',
    eta='auto',
    iterations=iterations,
    num_topics=num_topics,
    passes=passes,
    eval_every=eval_every
)

In [17]:
model.print_topics()

[(6,
  '0.034*"mutasi" + 0.023*"varian" + 0.009*"baru" + 0.008*"beralkohol" + 0.008*"nirawak" + 0.008*"wahid" + 0.008*"opec" + 0.007*"sim" + 0.007*"genom" + 0.007*"lebih"'),
 (104,
  '0.017*"candi" + 0.014*"magang" + 0.012*"veronica" + 0.010*"nduga" + 0.008*"ikea" + 0.008*"orang" + 0.008*"aidi" + 0.007*"kata" + 0.006*"lebih" + 0.006*"intan"'),
 (137,
  '0.230*"kucing" + 0.066*"boneka" + 0.046*"upacara" + 0.028*"post" + 0.023*"thai" + 0.022*"mainan" + 0.020*"may" + 0.020*"terimakasih" + 0.018*"freedom" + 0.017*"bayang"'),
 (177,
  '0.106*"swedia" + 0.062*"batu" + 0.025*"sari" + 0.024*"h" + 0.023*"stockholm" + 0.018*"collins" + 0.017*"bonus" + 0.015*"rambu" + 0.014*"stone" + 0.012*"nevada"'),
 (78,
  '0.042*"bunga" + 0.038*"suku" + 0.032*"ghana" + 0.022*"gardner" + 0.022*"maraton" + 0.020*"linda" + 0.013*"nintendo" + 0.012*"fajar" + 0.010*"salon" + 0.010*"peti"'),
 (151,
  '0.186*"foto" + 0.070*"bulan" + 0.027*"armstrong" + 0.016*"neil" + 0.015*"radiasi" + 0.013*"apollo" + 0.013*"sabuk" 

In [18]:
model.num_topics

200

In [19]:
from gensim.models import CoherenceModel

# Calculate Coherence Score
coherence_model_lda = CoherenceModel(model=model, texts=proc_train_docs, dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()

print(f"Coherence Score for LDA: {coherence_lda}")

Coherence Score for LDA: 0.44904604860841446


In [20]:
# Number of top words to consider for each topic
top_n = 10

# Extract the top words for each topic
topic_words = []
for t in range(num_topics):
    top_words = model.show_topic(t, topn=top_n)
    topic_words.append([word for word, _ in top_words])

In [21]:
# Collect all top words from all topics
unique_words = set()
total_words = 0

for words in topic_words:
    total_words += len(words)
    for word in words:
        unique_words.add(word)

# Number of unique words across all topics
num_unique_words = len(unique_words)

# Topic Diversity
topic_diversity = num_unique_words / total_words

print(f"Topic Diversity for LDA: {topic_diversity:.4f}")


Topic Diversity for LDA: 0.6865


In [22]:
model.save('lda.model')

In [23]:
# later on, load trained model from file
loaded_model =  LdaModel.load('lda.model')

In [30]:
loaded_model[val_bow[0]]

[(5, 0.021407016),
 (44, 0.19231427),
 (49, 0.021714589),
 (53, 0.03147292),
 (55, 0.073937505),
 (77, 0.06466677),
 (82, 0.079851836),
 (117, 0.05097067),
 (134, 0.039187554),
 (137, 0.037990928),
 (157, 0.10843223),
 (169, 0.037358895),
 (176, 0.22711481)]

In [33]:
print(proc_val_docs[0])

['dilaporkan', 'orang', 'terluka', 'cukup', 'serius', 'sisanya', 'diperbolehkan', 'pulang', 'mendapatkan', 'perawatan', 'video', 'gajah', 'mengamuk', 'kontan', 'viral', 'media', 'sosial', 'si', 'gajah', 'berlari', 'tak', 'arah', 'menabrak', 'menginjak', 'sebagian', 'peserta', 'upacara', 'simak', 'diketahui', 'penyebab', 'gajah', 'tersebut', 'tiba', 'tiba', 'mengamuk', 'diduga', 'gajah', 'kaget', 'sesuatu', 'antara', 'peserta', 'pengunjung', 'media', 'setempat', 'melaporkan', 'gajah', 'yang', 'mengamuk', 'prosesi', 'berbeda', 'gajah', 'hias', 'merupakan', 'daya', 'tarik', 'tersendiri', 'upacara', 'keagamaan', 'sri', 'lanka', 'warga', 'sri', 'lanka', 'memiliki', 'gajah', 'simbol', 'status', 'beberapa', 'kuil', 'sri', 'lanka', 'memiliki', 'gajah', 'keperluan', 'upacara']


In [40]:
# print topic 28
loaded_model.print_topic(topicno=157)

'0.124*"iran" + 0.019*"mugabe" + 0.018*"teheran" + 0.017*"zimbabwe" + 0.016*"rouhani" + 0.016*"hassan" + 0.016*"lanka" + 0.014*"sri" + 0.012*"revolusi" + 0.011*"jamur"'